In [2]:
from homework.smoke_test import SAMPLE_URL

PREFIX = "https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle"
!curl -fL -o hair_classifier_v1.onnx.data "{PREFIX}/hair_classifier_v1.onnx.data"
!curl -fL -o hair_classifier_v1.onnx "{PREFIX}/hair_classifier_v1.onnx"
!curl -fL -o sample.jpeg \
  "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"


FileNotFoundError: [Errno 2] No such file or directory: 'model.bin'

In [3]:
!sha256sum hair_classifier_v1.onnx hair_classifier_v1.onnx.data sample.jpeg

2b1adcb51745b73609ac7efebf3b6199483a931ed6dfe35ab925978f4357c2d8  hair_classifier_v1.onnx
6ba88582d6098a535f918d9a56ad23800f919ad7bd5aad67e5b2bb3e289e5a7e  hair_classifier_v1.onnx.data
6b3ae4003b5b6d54b51ee4d5fc5f183126bb1be0bf7066e5a846785fdcbfd2f5  sample.jpeg


In [4]:
!uv venv --python 3.13 .venv
!uv pip install --python .venv -r requirements.txt


Using CPython 3.13.15
Creating virtual environment at: .venv
? A virtual environment already exists at `.venv`. Do you want to replace it? [y/n] › yes

hint: Use the `--clear` flag or set `UV_VENV_CLEAR=1` to skip this prompt
Checked 10 packages in 0.65ms


In [5]:
import onnxruntime as ort

In [6]:
onnx_model_path = 'hair_classifier_v1.onnx'
session = ort.InferenceSession(onnx_model_path, providers=['CPUExecutionProvider'])

In [7]:
inputs = session.get_inputs()
outputs = session.get_outputs()

input_name = inputs[0].name
output_name = outputs[0].name

In [8]:
input_name

'input'

In [9]:
output_name

'output'

In [10]:
from io import BytesIO
from urllib import request

import numpy as np
from PIL import Image


def download_image(url):
    with request.urlopen(url, timeout=30) as response:
        return Image.open(BytesIO(response.read())).convert("RGB")


def prepare_image(image):
    image = image.resize((200, 200), Image.Resampling.BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    array = (array - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array(
        [0.229, 0.224, 0.225], dtype=np.float32
    )
    return np.transpose(array, (2, 0, 1))[None, ...]

In [11]:
sample_img = 'sample.jpeg'
image = Image.open(sample_img).convert('RGB')
tensor = prepare_image(image)

r_channel_value = round(float(tensor[0, 0, 0, 0]), 3)
r_channel_value

-1.056

In [12]:
import onnxruntime as ort
SAMPLE_URL = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"
session = ort.InferenceSession(
    "hair_classifier_v1.onnx",
    providers=["CPUExecutionProvider"],
)
output = session.run(
    ["output"],
    {"input": prepare_image(download_image(SAMPLE_URL))},
)[0]

In [13]:
output

array([[0.72769666]], dtype=float32)

In [14]:
round(output[0][0],3)

np.float32(0.728)

In [15]:
!python smoke_test.py

{'straight_probability': 0.728, 'straight': True}


In [16]:
!podman build -t mlzoomcamp-2026-serverless .


STEP 1/5: FROM public.ecr.aws/lambda/python:3.13@sha256:0fc65c4c9bace18ce7cb5b670119b847140a9c1c45f2b24d06ce4e35bef0a9be
STEP 2/5: COPY requirements-lambda.txt ${LAMBDA_TASK_ROOT}/
--> Using cache 9f77ea29f2292d3a5c16f2e31a32e350d94b2e0c3ac0f7e1842515d68baaad5f
--> 9f77ea29f229
STEP 3/5: RUN python -m pip install --no-cache-dir -r ${LAMBDA_TASK_ROOT}/requirements-lambda.txt     --target ${LAMBDA_TASK_ROOT}
--> Using cache cddb85d9bd43ba0fd4f304e1d2d4b93e9c4422395fd2ec5f6337d49a0c03418d
--> cddb85d9bd43
STEP 4/5: COPY lambda_function.py hair_classifier_v1.onnx hair_classifier_v1.onnx.data ${LAMBDA_TASK_ROOT}/
--> Using cache c6d67bd4c2a36991e39afc10345b03017057cbbd36d50b209f4e80aa8bada8ec
--> c6d67bd4c2a3
STEP 5/5: CMD ["lambda_function.lambda_handler"]
--> Using cache 81e58482af22b31091cbf4e7686f9eccb3376cbb65e7c7611e31c8f5b58b1e57
COMMIT mlzoomcamp-2026-serverless
--> 81e58482af22
Successfully tagged localhost/mlzoomcamp-2026-serverless:latest
81e58482af22b31091cbf4e7686f9eccb3376cbb6

In [17]:
!podman run -d --rm -p 9000:8080 --name hair-classifier mlzoomcamp-2026-serverless


Error: creating container storage: the container name "hair-classifier" is already in use by 0149eb951183bd80fec3fce6358d6f773f6786740c3f857e808ac7d3bf8c52e6. You have to remove that container to be able to reuse that name: that name is already in use, or use --replace to instruct Podman to do so.


In [18]:
!curl -s \
  -XPOST 'http://localhost:9000/2015-03-31/functions/function/invocations' \
  -H 'Content-Type: application/json' \
  -d '{"image_url":"https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"}'

{"statusCode": 200, "headers": {"Content-Type": "application/json"}, "body": "{\"straight_probability\": 0.728, \"straight\": true}"}